# Kiểm tra Phân phối Nhãn qua các bước xử lý

Notebook này kiểm tra và so sánh phân phối nhãn (`label_id`) của bộ dữ liệu ViHSD qua 3 bước:
1. **Raw** — dữ liệu gốc (`train.csv`, `dev.csv`, `test.csv`)
2. **Structural Clean** — sau khi xử lý null và duplicates (`*_structural_clean.csv`)
3. **NLP Clean** — sau khi tiền xử lý ngôn ngữ (`*_clean.csv`)

Kết quả dùng để điền số liệu vào báo cáo.

In [16]:
import pandas as pd

SPLITS = ['train', 'dev', 'test']
LABEL_NAMES = {0: 'CLEAN', 1: 'OFFENSIVE', 2: 'HATE'}
DATA_DIR = '../data'

## 1. Dữ liệu gốc (Raw)

In [17]:
print('=' * 60)
print('RAW DATA')
print('=' * 60)

raw_stats = {}
for split in SPLITS:
    df = pd.read_csv(f'{DATA_DIR}/{split}.csv')
    dist = df['label_id'].value_counts().sort_index().to_dict()
    raw_stats[split] = {'total': len(df), 'dist': dist}
    print(f'\n{split}.csv:')
    print(f'  Tổng dòng (raw lines): ', end='')
    with open(f'{DATA_DIR}/{split}.csv', 'r', encoding='utf-8') as f:
        print(sum(1 for _ in f))
    print(f'  Tổng mẫu (pandas):     {len(df)}')
    print(f'  Null (free_text):      {df["free_text"].isnull().sum()}')
    print(f'  Duplicates:            {df.duplicated().sum()}')
    print(f'  Phân phối nhãn:')
    for label_id, count in dist.items():
        pct = count / len(df) * 100
        print(f'    [{label_id}] {LABEL_NAMES[label_id]:12s}: {count:6,d} ({pct:.1f}%)')

RAW DATA

train.csv:
  Tổng dòng (raw lines): 24776
  Tổng mẫu (pandas):     24048
  Null (free_text):      2
  Duplicates:            1356
  Phân phối nhãn:
    [0] CLEAN       : 19,886 (82.7%)
    [1] OFFENSIVE   :  1,606 (6.7%)
    [2] HATE        :  2,556 (10.6%)

dev.csv:
  Tổng dòng (raw lines): 2680
  Tổng mẫu (pandas):     2672
  Null (free_text):      0
  Duplicates:            21
  Phân phối nhãn:
    [0] CLEAN       :  2,190 (82.0%)
    [1] OFFENSIVE   :    212 (7.9%)
    [2] HATE        :    270 (10.1%)

test.csv:
  Tổng dòng (raw lines): 6692
  Tổng mẫu (pandas):     6680
  Null (free_text):      0
  Duplicates:            98
  Phân phối nhãn:
    [0] CLEAN       :  5,548 (83.1%)
    [1] OFFENSIVE   :    444 (6.6%)
    [2] HATE        :    688 (10.3%)


## 2. Sau làm sạch cấu trúc (Structural Clean)

In [18]:
print('=' * 60)
print('SAU STRUCTURAL CLEAN (*_structural_clean.csv)')
print('=' * 60)

structural_stats = {}
for split in SPLITS:
    df = pd.read_csv(f'{DATA_DIR}/{split}_structural_clean.csv')
    dist = df['label_id'].value_counts().sort_index().to_dict()
    structural_stats[split] = {'total': len(df), 'dist': dist}
    removed = raw_stats[split]['total'] - len(df)
    print(f'\n{split}_structural_clean.csv:')
    print(f'  Tổng mẫu:  {len(df):,}')
    print(f'  Đã loại:   {removed:,} ({removed/raw_stats[split]["total"]*100:.1f}%)')
    print(f'  Phân phối nhãn:')
    for label_id, count in dist.items():
        prev = raw_stats[split]['dist'].get(label_id, 0)
        pct = count / len(df) * 100
        print(f'    [{label_id}] {LABEL_NAMES[label_id]:12s}: {count:6,d} ({pct:.1f}%)  [thay đổi: {count - prev:+d}]')

SAU STRUCTURAL CLEAN (*_structural_clean.csv)

train_structural_clean.csv:
  Tổng mẫu:  22,690
  Đã loại:   1,358 (5.6%)
  Phân phối nhãn:
    [0] CLEAN       : 18,652 (82.2%)  [thay đổi: -1234]
    [1] OFFENSIVE   :  1,543 (6.8%)  [thay đổi: -63]
    [2] HATE        :  2,495 (11.0%)  [thay đổi: -61]

dev_structural_clean.csv:
  Tổng mẫu:  2,651
  Đã loại:   21 (0.8%)
  Phân phối nhãn:
    [0] CLEAN       :  2,171 (81.9%)  [thay đổi: -19]
    [1] OFFENSIVE   :    210 (7.9%)  [thay đổi: -2]
    [2] HATE        :    270 (10.2%)  [thay đổi: +0]

test_structural_clean.csv:
  Tổng mẫu:  6,582
  Đã loại:   98 (1.5%)
  Phân phối nhãn:
    [0] CLEAN       :  5,461 (83.0%)  [thay đổi: -87]
    [1] OFFENSIVE   :    440 (6.7%)  [thay đổi: -4]
    [2] HATE        :    681 (10.3%)  [thay đổi: -7]


## 3. Sau tiền xử lý NLP (NLP Clean)

In [19]:
print('=' * 60)
print('SAU NLP PREPROCESSING (*_clean.csv)')
print('=' * 60)

nlp_stats = {}
for split in SPLITS:
    df = pd.read_csv(f'{DATA_DIR}/{split}_clean.csv')
    dist = df['label_id'].value_counts().sort_index().to_dict()
    avg_len = df['free_text_clean'].str.len().mean()
    avg_words = df['free_text_clean'].str.split().str.len().mean()
    nlp_stats[split] = {'total': len(df), 'dist': dist, 'avg_len': avg_len, 'avg_words': avg_words}
    removed = structural_stats[split]['total'] - len(df)
    print(f'\n{split}_clean.csv:')
    print(f'  Tổng mẫu:          {len(df):,}')
    print(f'  Đã loại (sau NLP): {removed:,}')
    print(f'  Độ dài TB:         {avg_len:.1f} ký tự')
    print(f'  Số từ TB:          {avg_words:.1f} từ')
    print(f'  Phân phối nhãn:')
    for label_id, count in dist.items():
        prev = structural_stats[split]['dist'].get(label_id, 0)
        pct = count / len(df) * 100
        print(f'    [{label_id}] {LABEL_NAMES[label_id]:12s}: {count:6,d} ({pct:.1f}%)  [thay đổi: {count - prev:+d}]')

SAU NLP PREPROCESSING (*_clean.csv)

train_clean.csv:
  Tổng mẫu:          22,510
  Đã loại (sau NLP): 180
  Độ dài TB:         48.1 ký tự
  Số từ TB:          11.7 từ
  Phân phối nhãn:
    [0] CLEAN       : 18,473 (82.1%)  [thay đổi: -179]
    [1] OFFENSIVE   :  1,542 (6.9%)  [thay đổi: -1]
    [2] HATE        :  2,495 (11.1%)  [thay đổi: +0]

dev_clean.csv:
  Tổng mẫu:          2,633
  Đã loại (sau NLP): 18
  Độ dài TB:         47.1 ký tự
  Số từ TB:          11.5 từ
  Phân phối nhãn:
    [0] CLEAN       :  2,153 (81.8%)  [thay đổi: -18]
    [1] OFFENSIVE   :    210 (8.0%)  [thay đổi: +0]
    [2] HATE        :    270 (10.3%)  [thay đổi: +0]

test_clean.csv:
  Tổng mẫu:          6,527
  Đã loại (sau NLP): 55
  Độ dài TB:         47.4 ký tự
  Số từ TB:          11.5 từ
  Phân phối nhãn:
    [0] CLEAN       :  5,406 (82.8%)  [thay đổi: -55]
    [1] OFFENSIVE   :    440 (6.7%)  [thay đổi: +0]
    [2] HATE        :    681 (10.4%)  [thay đổi: +0]


## 4. Bảng tổng hợp

In [20]:
rows = []
for label_id, label_name in LABEL_NAMES.items():
    row = {'Nhãn': f'{label_name} ({label_id})'}
    for split in SPLITS:
        row[f'{split}_raw']        = raw_stats[split]['dist'].get(label_id, 0)
        row[f'{split}_structural'] = structural_stats[split]['dist'].get(label_id, 0)
        row[f'{split}_nlp']        = nlp_stats[split]['dist'].get(label_id, 0)
    rows.append(row)

# Hàng tổng
total_row = {'Nhãn': 'TỔNG'}
for split in SPLITS:
    total_row[f'{split}_raw']        = raw_stats[split]['total']
    total_row[f'{split}_structural'] = structural_stats[split]['total']
    total_row[f'{split}_nlp']        = nlp_stats[split]['total']
rows.append(total_row)

summary_df = pd.DataFrame(rows).set_index('Nhãn')

# Định dạng cột cho dễ đọc
multi_cols = pd.MultiIndex.from_tuples([
    ('Train', 'Raw'), ('Train', 'Structural'), ('Train', 'NLP'),
    ('Dev',   'Raw'), ('Dev',   'Structural'), ('Dev',   'NLP'),
    ('Test',  'Raw'), ('Test',  'Structural'), ('Test',  'NLP'),
])
summary_df.columns = multi_cols
print(summary_df.to_string())

               Train                     Dev                   Test                 
                 Raw Structural    NLP   Raw Structural   NLP   Raw Structural   NLP
Nhãn                                                                                
CLEAN (0)      19886      18652  18473  2190       2171  2153  5548       5461  5406
OFFENSIVE (1)   1606       1543   1542   212        210   210   444        440   440
HATE (2)        2556       2495   2495   270        270   270   688        681   681
TỔNG           24048      22690  22510  2672       2651  2633  6680       6582  6527


## 5. Đảm bảo chất lượng dữ liệu (Quality Assurance)

Kiểm tra toàn diện trên tập dữ liệu cuối cùng (`*_clean.csv`) theo các tiêu chí:
- **Tính đầy đủ:** Không còn null hoặc chuỗi rỗng
- **Tính duy nhất:** Không còn trùng lặp
- **Tính hợp lệ:** Tất cả nhãn thuộc {0, 1, 2}
- **Phát hiện ngoại lệ:** Mẫu có độ dài bất thường (> 3σ)
- **Tính nhất quán:** Phân phối nhãn ổn định qua các tập

In [21]:
print('=' * 65)
print('KIỂM TRA CHẤT LƯỢNG DỮ LIỆU CUỐI CÙNG (*_clean.csv)')
print('=' * 65)

qa_results = {}
for split in SPLITS:
    df = pd.read_csv(f'{DATA_DIR}/{split}_clean.csv')
    text = df['free_text_clean']
    lengths = text.str.len()
    words = text.str.split().str.len()
    threshold_3sigma = lengths.mean() + 3 * lengths.std()
    outliers = (lengths > threshold_3sigma).sum()
    very_short = (words < 3).sum()
    valid_labels = df['label_id'].isin([0, 1, 2]).all()

    qa_results[split] = {
        'total': len(df),
        'null': text.isnull().sum(),
        'empty': (text == '').sum(),
        'dup': df.duplicated().sum(),
        'len_min': lengths.min(), 'len_max': lengths.max(),
        'len_mean': round(lengths.mean(), 1), 'len_median': lengths.median(),
        'word_min': words.min(), 'word_max': words.max(),
        'word_mean': round(words.mean(), 1), 'word_median': words.median(),
        'outliers': outliers, 'very_short': very_short,
        'valid_labels': valid_labels
    }

    status = lambda x: '✅' if x == 0 else f'⚠️  {x}'
    print(f'\n--- {split}_clean.csv ({len(df):,} mẫu) ---')
    print(f'  Null còn lại:         {status(text.isnull().sum())}')
    print(f'  Chuỗi rỗng:           {status((text == "").sum())}')
    print(f'  Trùng lặp:            {status(df.duplicated().sum())}')
    print(f'  Nhãn hợp lệ [0,1,2]:  {"✅ Tất cả" if valid_labels else "❌ Có nhãn lạ"}')
    print(f'  Độ dài (ký tự): min={lengths.min()}, max={lengths.max()}, mean={lengths.mean():.1f}, median={lengths.median():.0f}')
    print(f'  Số từ:          min={words.min()}, max={words.max()}, mean={words.mean():.1f}, median={words.median():.0f}')
    print(f'  Outliers (>3σ={threshold_3sigma:.0f} ký tự): {outliers} mẫu ({outliers/len(df)*100:.2f}%)')
    print(f'  Rất ngắn (<3 từ):     {very_short} mẫu ({very_short/len(df)*100:.1f}%)')


KIỂM TRA CHẤT LƯỢNG DỮ LIỆU CUỐI CÙNG (*_clean.csv)

--- train_clean.csv (22,510 mẫu) ---
  Null còn lại:         ✅
  Chuỗi rỗng:           ✅
  Trùng lặp:            ✅
  Nhãn hợp lệ [0,1,2]:  ✅ Tất cả
  Độ dài (ký tự): min=1, max=7084, mean=48.1, median=33
  Số từ:          min=1, max=1550, mean=11.7, median=8
  Outliers (>3σ=301 ký tự): 108 mẫu (0.48%)
  Rất ngắn (<3 từ):     1460 mẫu (6.5%)

--- dev_clean.csv (2,633 mẫu) ---
  Null còn lại:         ✅
  Chuỗi rỗng:           ✅
  Trùng lặp:            ✅
  Nhãn hợp lệ [0,1,2]:  ✅ Tất cả
  Độ dài (ký tự): min=1, max=573, mean=47.1, median=33
  Số từ:          min=1, max=128, mean=11.5, median=8
  Outliers (>3σ=195 ký tự): 65 mẫu (2.47%)
  Rất ngắn (<3 từ):     211 mẫu (8.0%)

--- test_clean.csv (6,527 mẫu) ---
  Null còn lại:         ✅
  Chuỗi rỗng:           ✅
  Trùng lặp:            ✅
  Nhãn hợp lệ [0,1,2]:  ✅ Tất cả
  Độ dài (ký tự): min=1, max=1762, mean=47.4, median=33
  Số từ:          min=1, max=411, mean=11.5, median=8
  Outliers

In [22]:
# Bảng QA tổng hợp
print('\n--- BẢNG TỔNG HỢP KIỂM TRA CHẤT LƯỢNG ---')
print(f'{"Tiêu chí":<30} {"Train":>10} {"Dev":>10} {"Test":>10}')
print('-' * 65)

criteria = [
    ('Tổng mẫu', 'total', '{}'),
    ('Null còn lại', 'null', '{}'),
    ('Chuỗi rỗng', 'empty', '{}'),
    ('Trùng lặp', 'dup', '{}'),
    ('Độ dài min (ký tự)', 'len_min', '{}'),
    ('Độ dài max (ký tự)', 'len_max', '{}'),
    ('Độ dài mean (ký tự)', 'len_mean', '{:.1f}'),
    ('Số từ mean', 'word_mean', '{:.1f}'),
    ('Outliers >3σ', 'outliers', '{}'),
    ('Rất ngắn <3 từ', 'very_short', '{}'),
]

for name, key, fmt in criteria:
    vals = [fmt.format(qa_results[s][key]) for s in SPLITS]
    print(f'{name:<30} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10}')



--- BẢNG TỔNG HỢP KIỂM TRA CHẤT LƯỢNG ---
Tiêu chí                            Train        Dev       Test
-----------------------------------------------------------------
Tổng mẫu                            22510       2633       6527
Null còn lại                            0          0          0
Chuỗi rỗng                              0          0          0
Trùng lặp                               0          0          0
Độ dài min (ký tự)                      1          1          1
Độ dài max (ký tự)                   7084        573       1762
Độ dài mean (ký tự)                  48.1       47.1       47.4
Số từ mean                           11.7       11.5       11.5
Outliers >3σ                          108         65        136
Rất ngắn <3 từ                       1460        211        470


## 6. Thống kê mô tả tổng quan

Bảng tổng hợp thông tin cơ bản của 3 tập và thống kê mô tả đầy đủ (mean, median, std, IQR, min, max, Q1, Q3) của độ dài văn bản.

In [23]:
import pandas as pd
import numpy as np

SPLITS = ['train', 'dev', 'test']
DATA_DIR = '../data'
dfs = {s: pd.read_csv(f'{DATA_DIR}/{s}_clean.csv') for s in SPLITS}

# --- Bảng 1: Tổng quan các tập ---
print('=== BẢNG TỔNG QUAN CÁC TẬP DỮ LIỆU ===')
print(f'{"Tập":<8} {"Số mẫu":>10} {"Số cột":>8} {"Giá trị null":>14} {"Bản ghi trùng":>16}')
print('-' * 60)
total = 0
for s in SPLITS:
    df = dfs[s]
    total += len(df)
    print(f'{s:<8} {len(df):>10,} {df.shape[1]:>8} {df.isnull().sum().sum():>14} {df.duplicated().sum():>16}')
print('-' * 60)
print(f'{"Tổng":<8} {total:>10,}')  

# --- Bảng 2: Thống kê mô tả độ dài văn bản (tập train) ---
print('\n=== THỐNG KÊ MÔ TẢ ĐỘ DÀI VĂN BẢN (TẬP TRAIN) ===')
train = dfs['train'].copy()
train['char_len'] = train['free_text_clean'].astype(str).str.len()
train['word_count'] = train['free_text_clean'].astype(str).str.split().str.len()

stats = {}
for col, label in [('char_len', 'Độ dài (ký tự)'), ('word_count', 'Số từ')]:
    s = train[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    stats[label] = {
        'Số mẫu': len(s),
        'Min': s.min(),
        'Max': s.max(),
        'Mean': round(s.mean(), 1),
        'Median': s.median(),
        'Std': round(s.std(), 1),
        'Q1 (25%)': int(q1),
        'Q3 (75%)': int(q3),
        'IQR': int(q3 - q1),
    }

stats_df = pd.DataFrame(stats)
print(stats_df.to_string())
print(f'\nGhi chú: std/mean(char_len) = {train["char_len"].std()/train["char_len"].mean():.2f} (phân bố lệch phải rõ rệt)')


=== BẢNG TỔNG QUAN CÁC TẬP DỮ LIỆU ===
Tập          Số mẫu   Số cột   Giá trị null    Bản ghi trùng
------------------------------------------------------------
train        22,510        3              0                0
dev           2,633        3              0                0
test          6,527        3              0                0
------------------------------------------------------------
Tổng         31,670

=== THỐNG KÊ MÔ TẢ ĐỘ DÀI VĂN BẢN (TẬP TRAIN) ===
          Độ dài (ký tự)    Số từ
Số mẫu           22510.0  22510.0
Min                  1.0      1.0
Max               7084.0   1550.0
Mean                48.1     11.7
Median              33.0      8.0
Std                 84.2     19.4
Q1 (25%)            19.0      5.0
Q3 (75%)            56.0     14.0
IQR                 37.0      9.0

Ghi chú: std/mean(char_len) = 1.75 (phân bố lệch phải rõ rệt)
